In [1]:
import pandas as pd
from PIL import Image

In [2]:
# Read the file and load into a DataFrame
ground_truth_df = pd.read_csv('ml_exercise_therapanacea/label_train.txt', header=None, names=['label'])
ground_truth_df.index = ground_truth_df.index + 1

In [3]:
# all_zeros = ground_truth_df[ground_truth_df['label'] == 0].index.to_list()
# for x in all_zeros[100:105]:
#     img=Image.open(f'ml_exercise_therapanacea/train_img/{x:06d}.jpg')
#     display(img)

# all_ones = ground_truth_df[ground_truth_df['label'] == 1].index.to_list()
# for x in all_ones[100:105]:
#     img=Image.open(f'ml_exercise_therapanacea/train_img/{x:06d}.jpg')
#     display(img)

In [4]:
import torchvision
from torchvision.models import EfficientNet_B0_Weights
import torch
m = torchvision.models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
m.classifier = torch.nn.Linear(in_features=1280, out_features=2, bias=True)
m

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [5]:
proper_transforms = EfficientNet_B0_Weights.IMAGENET1K_V1.transforms()
proper_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [6]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class CelebASubsetDataset(Dataset):
    def __init__(self, img_folder, ground_truth_df, indices=None, transform=None):
        self.img_folder = img_folder
        self.ground_truth_df = ground_truth_df
        self.transform = transform
        if indices is None:
            self.indices = self.ground_truth_df.index.tolist()
        else:
            self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img_idx = self.indices[idx]
        img_path = f"{self.img_folder}/{img_idx:06d}.jpg"
        image = Image.open(img_path).convert("RGB")
        label = self.ground_truth_df.loc[img_idx, 'label']
        if self.transform:
            transf_image = self.transform(image)
            return transf_image, label
        return image, label

In [7]:
training_ds = CelebASubsetDataset(
    img_folder='ml_exercise_therapanacea/train_img',
    ground_truth_df=ground_truth_df,
    indices=list(range(1, 8000)),
    transform=proper_transforms
)

validation_ds = CelebASubsetDataset(
    img_folder='ml_exercise_therapanacea/train_img',
    ground_truth_df=ground_truth_df,
    indices=list(range(8001, 10000)),
    transform=proper_transforms
)

In [8]:
m.eval()
for x in range(len(validation_ds)):
    transf_img, label = validation_ds[x]
    # display(img)
    model_output = m(transf_img.unsqueeze(0))
    print(model_output)
    print(torch.argmax(model_output, dim=1).item())
    A, B = torch.max(model_output, 1)
    print(A,B)
    break

tensor([[-0.1519, -0.0813]], grad_fn=<AddmmBackward0>)
1
tensor([-0.0813], grad_fn=<MaxBackward0>) tensor([1])


In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim


# Hyperparameters
batch_size = 32
num_epochs = 5
learning_rate = 1e-3

# DataLoader
train_loader = DataLoader(training_ds, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_ds, batch_size=batch_size, shuffle=False)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(m.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    m.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for transf_images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()
        outputs = m(transf_images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total

    # Validation
    m.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for val_images, val_labels in tqdm(validation_loader, desc="Validation"):
            val_outputs = m(val_images)
            v_loss = criterion(val_outputs, val_labels)
            val_loss += v_loss.item() * val_labels.size(0)
            val_pred = torch.argmax(val_outputs, dim=1)
            val_correct += (val_pred == val_labels).sum().item()
            val_total += val_labels.size(0)
    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Validation: 100%|██████████| 63/63 [02:00<00:00,  1.91s/it]


Epoch 1/5, Train Loss: 0.2402, Train Acc: 0.9094, Val Loss: 0.1993, Val Acc: 0.9295


Validation: 100%|██████████| 63/63 [01:58<00:00,  1.88s/it]


Epoch 2/5, Train Loss: 0.1658, Train Acc: 0.9339, Val Loss: 0.1646, Val Acc: 0.9315


Validation: 100%|██████████| 63/63 [01:57<00:00,  1.87s/it]


Epoch 3/5, Train Loss: 0.1394, Train Acc: 0.9406, Val Loss: 0.1547, Val Acc: 0.9370


Validation: 100%|██████████| 63/63 [01:57<00:00,  1.86s/it]


Epoch 4/5, Train Loss: 0.1213, Train Acc: 0.9489, Val Loss: 0.1556, Val Acc: 0.9370


Validation: 100%|██████████| 63/63 [02:09<00:00,  2.05s/it]

Epoch 5/5, Train Loss: 0.1103, Train Acc: 0.9541, Val Loss: 0.1892, Val Acc: 0.9350


In [10]:
# save the model
torch.save(m.state_dict(), 'baseline.pth')